<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/Model13.11F.44.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ===============================
# Model 13 - 11 feature - reproducible
# ===============================

!pip install -q gdown

import gdown
gdown.download(
    id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
    output="HPV2025.xlsx",
    quiet=False
)

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE


# -------------------------------
# Reproducibility
# -------------------------------

def seed_everything(seed=44):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    np.random.seed(seed)
    random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    torch.use_deterministic_algorithms(True, warn_only=True)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 44
seed_everything(SEED)


# -------------------------------
# Read Dataset
# -------------------------------

df = pd.read_excel("HPV2025.xlsx")

df = df.dropna(subset=["HPV Status"])


# -------------------------------
# Missing values
# -------------------------------

tobacco_mode = df["Tobacco Consumption"].mode()[0]
df["Tobacco Consumption"] = df["Tobacco Consumption"].fillna(tobacco_mode)

alcohol_mode = df["Alcohol Consumption"].mode()[0]
df["Alcohol Consumption"] = df["Alcohol Consumption"].fillna(alcohol_mode)


# -------------------------------
# Remove unwanted columns
# -------------------------------

df = df.drop(
    columns=[
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

df = df.dropna()

print(df.shape)


# -------------------------------
# Encode categorical variables
# -------------------------------

df["T-stage"] = df["T-stage"].replace({
    "T0":0,
    "T1":1,
    "T2":2,
    "T3":3,
    "T4":4
})

df["N-stage"] = df["N-stage"].replace({
    "N0":0,
    "N1":1,
    "N2":2,
    "N3":3
})

df["M-stage"] = df["M-stage"].replace({
    "M0":0,
    "M1":1
})


# -------------------------------
# Features
# -------------------------------

X = df[
    [
        "Age",
        "Gender",
        "Tobacco Consumption",
        "Alcohol Consumption",
        "Performance Status",
        "Relapse",
        "RFS",
        "Treatment",
        "T-stage",
        "N-stage",
        "M-stage"
    ]
]

y = df["HPV Status"]


# -------------------------------
# Train/Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED
)

print("Train samples:")
print(y_train.value_counts())

print("\nTest samples:")
print(y_test.value_counts())


# -------------------------------
# Standardisation
# -------------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# -------------------------------
# SMOTE
# -------------------------------

smote = SMOTE(random_state=SEED)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())


# -------------------------------
# Torch tensors
# -------------------------------

X_train_smote = torch.FloatTensor(X_train_smote)
X_test = torch.FloatTensor(X_test)

y_train_smote = torch.LongTensor(
    y_train_smote.to_numpy()
)

y_test = torch.LongTensor(
    y_test.to_numpy()
)

# -------------------------------
# Model 13 Architecture
# -------------------------------

class HPVNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(11, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 2)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)

        return x


# -------------------------------
# Model
# -------------------------------

model = HPVNet()

weights = torch.tensor(
    [3.0, 1.0],
    dtype=torch.float32
)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# -------------------------------
# Training
# -------------------------------

epochs = 500

best_test_loss = float("inf")
best_epoch = 0

for epoch in range(epochs):

    model.train()

    outputs = model(X_train_smote)

    train_loss = criterion(
        outputs,
        y_train_smote
    )

    optimizer.zero_grad()

    train_loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        test_outputs = model(X_test)

        test_loss = criterion(
            test_outputs,
            y_test
        )

    if test_loss.item() < best_test_loss:

        best_test_loss = test_loss.item()

        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

    if (epoch + 1) % 50 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={train_loss.item():.4f}, "
            f"Test={test_loss.item():.4f}"
        )


print("\nBest Test Loss =", round(best_test_loss, 4))
print("Best Epoch =", best_epoch)


# -------------------------------
# Load Best Model
# -------------------------------

model.load_state_dict(
    torch.load("best_model.pth")
)

model.eval()

with torch.no_grad():

    outputs = model(X_test)

    probabilities = torch.softmax(
        outputs,
        dim=1
    )

    predicted = torch.argmax(
        outputs,
        dim=1
    )


# -------------------------------
# Classification Report
# -------------------------------

print("\nClassification Report\n")

print(

    classification_report(

        y_test.numpy(),

        predicted.numpy(),

        digits=4

    )

)


# -------------------------------
# Confusion Matrix
# -------------------------------

cm = confusion_matrix(

    y_test.numpy(),

    predicted.numpy()

)

print("Confusion Matrix")

print(cm)


# -------------------------------
# Metrics
# -------------------------------

bal_acc = balanced_accuracy_score(

    y_test.numpy(),

    predicted.numpy()

)

f1 = f1_score(

    y_test.numpy(),

    predicted.numpy()

)

y_prob = probabilities.numpy()

auc = roc_auc_score(

    y_test.numpy(),

    y_prob[:, 1]

)

print(f"\nBalanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

print("\nArchitecture: 11-64-32-16-2")
print("Optimiser: Adam")
print("Learning Rate: 0.001")
print("Class Weights: [3.0, 1.0]")
print("SMOTE: Yes")
print("Epochs: 500")

Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 5.21MB/s]


(423, 12)
Train samples:
HPV Status
1.0    318
0.0     20
Name: count, dtype: int64

Test samples:
HPV Status
1.0    82
0.0     3
Name: count, dtype: int64

After SMOTE:
HPV Status
1.0    318
0.0    318
Name: count, dtype: int64


/tmp/ipykernel_2879/3369093939.py:107: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["T-stage"] = df["T-stage"].replace({
/tmp/ipykernel_2879/3369093939.py:115: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["N-stage"] = df["N-stage"].replace({
/tmp/ipykernel_2879/3369093939.py:122: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no

Epoch 50, Train=0.2952, Test=0.7492
Epoch 100, Train=0.1272, Test=0.6096
Epoch 150, Train=0.0679, Test=0.7190
Epoch 200, Train=0.0310, Test=0.8972
Epoch 250, Train=0.0146, Test=1.0491
Epoch 300, Train=0.0071, Test=1.1996
Epoch 350, Train=0.0037, Test=1.3514
Epoch 400, Train=0.0023, Test=1.4722
Epoch 450, Train=0.0017, Test=1.5740
Epoch 500, Train=0.0012, Test=1.6614

Best Test Loss = 0.5464
Best Epoch = 77

Classification Report

              precision    recall  f1-score   support

           0     0.0667    0.3333    0.1111         3
           1     0.9714    0.8293    0.8947        82

    accuracy                         0.8118        85
   macro avg     0.5190    0.5813    0.5029        85
weighted avg     0.9395    0.8118    0.8671        85

Confusion Matrix
[[ 1  2]
 [14 68]]

Balanced Accuracy: 0.5813
F1-score:          0.8947
AUC:               0.8618

Architecture: 11-64-32-16-2
Optimiser: Adam
Learning Rate: 0.001
Class Weights: [3.0, 1.0]
SMOTE: Yes
Epochs: 500
